# Stock Price Prediction: LSTM vs. Attention, with Walk-Forward Validation and BacktestingThis notebook builds, validates, and honestly evaluates two deep learning architectures for next-day stock price prediction:- **LSTM** — a 2-layer LSTM with dropout (the v1 baseline architecture)- **LSTM + Self-Attention** — an LSTM backbone with a multi-head attention layer over timestep outputs, letting the model learn which days in the lookback window matter most**What makes this version more rigorous than a typical tutorial notebook:**1. **Walk-forward (rolling-origin) cross-validation** instead of a single train/test split — the model is retrained and re-evaluated across multiple time periods, giving a realistic picture of performance variation across market regimes.2. **A naive baseline is always reported alongside the model** — "tomorrow's price = today's price." If a model can't beat this on a real test set, it hasn't learned anything beyond normal day-to-day autocorrelation.3. **A backtest with transaction costs**, not just price-prediction error metrics — RMSE doesn't tell you whether a strategy based on the model would actually make money after realistic trading costs.4. **Multi-ticker support** — the same pipeline runs across a basket of tickers, since a model that "works" on one stock by chance is far less convincing than one that's at least directionally consistent across several.5. **Proper scaler persistence** — the v1 implementation had a real bug where the live prediction API re-fit a new `MinMaxScaler` on each request instead of reusing the one fit during training. This is fixed throughout: see `src/sequences.py` and `src/train.py`.**Disclaimer:** This remains an educational project. Even with walk-forward validation and backtesting, **none of this should be used to make real investment decisions.** The point of the rigor added here is precisely to demonstrate, honestly, how hard it is for a model like this to beat a trivial baseline — which is itself an important and often-skipped lesson in applied time series ML.

## 1. Setup & Imports

In [ ]:
!pip install yfinance joblib -q

In [ ]:
import syssys.path.insert(0, '.')import numpy as npimport pandas as pdimport matplotlib.pyplot as pltfrom src.data_fetching import fetch_stock_data, fetch_multi_ticker_datafrom src.features import engineer_features, DEFAULT_FEATURE_COLSfrom src.sequences import chronological_split, fit_scaler, save_scaler, build_sequences, inverse_transform_columnfrom src.models import build_lstm_model, build_attention_modelfrom src.walk_forward import walk_forward_validate, summarize_foldsfrom src.backtest import backtest_directional_strategy, backtest_buy_and_holdimport warningswarnings.filterwarnings('ignore')np.random.seed(42)

## 2. ConfigurationAll key parameters live here. `TICKERS` controls which stocks are used for the multi-ticker comparison in Section 7; `PRIMARY_TICKER` is used for the detailed single-stock walkthrough in Sections 3–6.

In [ ]:
PRIMARY_TICKER = 'AAPL'TICKERS = ['AAPL', 'MSFT', 'GOOGL', 'JPM']   # used for the multi-ticker comparison in Section 7START_DATE = '2015-01-01'END_DATE = '2023-01-01'LOOKBACK_WINDOW = 60TRAIN_FRAC = 0.70VAL_FRAC = 0.15   # remaining 0.15 is testEPOCHS = 100BATCH_SIZE = 32# Walk-forward validation settingsWF_N_FOLDS = 5WF_MIN_TRAIN_SIZE = 600WF_TEST_SIZE = 90WF_EPOCHS = 30   # fewer epochs per fold since we're training n_folds modelsTRANSACTION_COST_BPS = 5.0   # round-trip cost per trade, in basis points

## 3. Data Fetching & Feature Engineering

In [ ]:
raw_data = fetch_stock_data(PRIMARY_TICKER, START_DATE, END_DATE)print(raw_data.head())

In [ ]:
featured = engineer_features(raw_data)featured[DEFAULT_FEATURE_COLS].tail()

### Visualise price and indicators

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)axes[0].plot(featured.index, featured['Adj Close'], label='Adj Close', color='black', linewidth=1)axes[0].plot(featured.index, featured['MA_short'], label='MA short', color='orange')axes[0].plot(featured.index, featured['MA_long'], label='MA long', color='blue')axes[0].set_title(f'{PRIMARY_TICKER} Price & Moving Averages')axes[0].legend()axes[1].plot(featured.index, featured['RSI'], color='purple')axes[1].axhline(70, color='red', linestyle='--', linewidth=0.8)axes[1].axhline(30, color='green', linestyle='--', linewidth=0.8)axes[1].set_title('RSI')axes[2].plot(featured.index, featured['MACD'], label='MACD', color='blue')axes[2].plot(featured.index, featured['MACD_signal'], label='Signal', color='red')axes[2].bar(featured.index, featured['MACD_hist'], label='Histogram', color='gray', alpha=0.3)axes[2].set_title('MACD')axes[2].legend()plt.tight_layout()plt.savefig('outputs_price_and_indicators.png', dpi=120)plt.show()

## 4. Train/Val/Test Split & Scaler Fitting**Critical detail:** the scaler is fit *only* on the training split, then reused (never re-fit) for validation and test data. This prevents test-set information from leaking into the scaling — a subtle form of data leakage that's easy to introduce accidentally (the v1 implementation actually had a version of this bug in its live API).

In [ ]:
train_df, val_df, test_df = chronological_split(featured, train_frac=TRAIN_FRAC, val_frac=VAL_FRAC)print(f"Train: {len(train_df)} rows ({train_df.index[0].date()} to {train_df.index[-1].date()})")print(f"Val:   {len(val_df)} rows ({val_df.index[0].date()} to {val_df.index[-1].date()})")print(f"Test:  {len(test_df)} rows ({test_df.index[0].date()} to {test_df.index[-1].date()})")scaler = fit_scaler(train_df, DEFAULT_FEATURE_COLS)save_scaler(scaler, f'models/{PRIMARY_TICKER}_lstm_scaler.joblib')X_train, y_train = build_sequences(train_df, DEFAULT_FEATURE_COLS, 'Adj Close', scaler, LOOKBACK_WINDOW)X_val, y_val = build_sequences(val_df, DEFAULT_FEATURE_COLS, 'Adj Close', scaler, LOOKBACK_WINDOW)X_test, y_test = build_sequences(test_df, DEFAULT_FEATURE_COLS, 'Adj Close', scaler, LOOKBACK_WINDOW)print(f"X_train: {X_train.shape}  X_val: {X_val.shape}  X_test: {X_test.shape}")

## 5. Train Both ArchitecturesWe train the baseline LSTM and the LSTM+Attention model on the same train/val split, so their performance is directly comparable.

In [ ]:
lstm_model = build_lstm_model((LOOKBACK_WINDOW, len(DEFAULT_FEATURE_COLS)))lstm_history = lstm_model.fit(    X_train, y_train,    validation_data=(X_val, y_val),    epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=1,)lstm_model.save(f'models/{PRIMARY_TICKER}_lstm.keras')

In [ ]:
attention_model = build_attention_model((LOOKBACK_WINDOW, len(DEFAULT_FEATURE_COLS)))attention_history = attention_model.fit(    X_train, y_train,    validation_data=(X_val, y_val),    epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=1,)attention_model.save(f'models/{PRIMARY_TICKER}_attention.keras')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))axes[0].plot(lstm_history.history['loss'], label='Train')axes[0].plot(lstm_history.history['val_loss'], label='Validation')axes[0].set_title('LSTM Training Loss')axes[0].legend()axes[1].plot(attention_history.history['loss'], label='Train')axes[1].plot(attention_history.history['val_loss'], label='Validation')axes[1].set_title('LSTM+Attention Training Loss')axes[1].legend()plt.tight_layout()plt.savefig('outputs_training_loss.png', dpi=120)plt.show()

## 6. Evaluate on Test Set — Model vs. Naive BaselineThis is the step most stock-prediction tutorials skip. We compare both models against the naive baseline of "tomorrow's price equals today's price." A model with impressive-looking RMSE that still loses to this baseline has not actually learned a useful pattern.

In [ ]:
target_idx = DEFAULT_FEATURE_COLS.index('Adj Close')def evaluate(model, X_test, y_test, name):    pred_scaled = model.predict(X_test, verbose=0).flatten()    pred_price = inverse_transform_column(pred_scaled, scaler, target_idx, len(DEFAULT_FEATURE_COLS))    actual_price = inverse_transform_column(y_test, scaler, target_idx, len(DEFAULT_FEATURE_COLS))    baseline_scaled = np.roll(y_test, 1)    baseline_scaled[0] = y_test[0]    baseline_price = inverse_transform_column(baseline_scaled, scaler, target_idx, len(DEFAULT_FEATURE_COLS))    from sklearn.metrics import mean_squared_error, mean_absolute_error    rmse = np.sqrt(mean_squared_error(actual_price, pred_price))    mae = mean_absolute_error(actual_price, pred_price)    mape = np.mean(np.abs((actual_price - pred_price) / actual_price)) * 100    b_rmse = np.sqrt(mean_squared_error(actual_price, baseline_price))    b_mae = mean_absolute_error(actual_price, baseline_price)    b_mape = np.mean(np.abs((actual_price - baseline_price) / actual_price)) * 100    print(f"=== {name} ===")    print(f"  Model:    RMSE=${rmse:.2f}  MAE=${mae:.2f}  MAPE={mape:.2f}%")    print(f"  Baseline: RMSE=${b_rmse:.2f}  MAE=${b_mae:.2f}  MAPE={b_mape:.2f}%")    print(f"  Result:   {'Model BEATS baseline' if rmse < b_rmse else 'Model LOSES to baseline'}")    return actual_price, pred_price, baseline_pricelstm_actual, lstm_pred, lstm_baseline = evaluate(lstm_model, X_test, y_test, 'LSTM')print()attn_actual, attn_pred, attn_baseline = evaluate(attention_model, X_test, y_test, 'LSTM + Attention')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)axes[0].plot(lstm_actual, color='black', label='Actual', linewidth=1.5)axes[0].plot(lstm_pred, color='blue', label='LSTM Prediction', alpha=0.8)axes[0].plot(lstm_baseline, color='gray', label='Naive Baseline', linestyle='--', alpha=0.6)axes[0].set_title(f'{PRIMARY_TICKER} — LSTM vs. Actual vs. Naive Baseline (Test Set)')axes[0].legend()axes[1].plot(attn_actual, color='black', label='Actual', linewidth=1.5)axes[1].plot(attn_pred, color='green', label='LSTM+Attention Prediction', alpha=0.8)axes[1].plot(attn_baseline, color='gray', label='Naive Baseline', linestyle='--', alpha=0.6)axes[1].set_title(f'{PRIMARY_TICKER} — LSTM+Attention vs. Actual vs. Naive Baseline (Test Set)')axes[1].legend()plt.tight_layout()plt.savefig('outputs_predictions_vs_actual.png', dpi=120)plt.show()

## 7. Walk-Forward Cross-ValidationA single train/test split tells you how the model performed on one particular stretch of history. Walk-forward validation retrains the model across multiple expanding windows and reports performance per fold — revealing whether the model's apparent skill is consistent or just a lucky test period.**Note:** this retrains a fresh model per fold, so it is meaningfully slower than a single train/test split. `WF_EPOCHS` is set lower than the main training run above to keep runtime reasonable; for a final/production evaluation, consider increasing it.

In [ ]:
wf_results = walk_forward_validate(    df=featured,    feature_cols=DEFAULT_FEATURE_COLS,    target_col='Adj Close',    lookback=LOOKBACK_WINDOW,    n_folds=WF_N_FOLDS,    min_train_size=WF_MIN_TRAIN_SIZE,    test_size=WF_TEST_SIZE,    build_model_fn=lambda: build_lstm_model((LOOKBACK_WINDOW, len(DEFAULT_FEATURE_COLS))),    epochs=WF_EPOCHS,    batch_size=BATCH_SIZE,    verbose=0,)wf_summary = summarize_folds(wf_results)wf_summary

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))x = wf_summary['fold']width = 0.35ax.bar(x - width/2, wf_summary['model_rmse'], width, label='Model RMSE')ax.bar(x + width/2, wf_summary['baseline_rmse'], width, label='Baseline RMSE')ax.set_xlabel('Fold')ax.set_ylabel('RMSE ($)')ax.set_title('Walk-Forward Validation: Model vs. Baseline RMSE per Fold')ax.legend()plt.savefig('outputs_walk_forward_rmse.png', dpi=120)plt.show()n_folds_beating_baseline = wf_summary['beats_baseline'].sum()print(f"Model beat the naive baseline in {n_folds_beating_baseline} out of {len(wf_summary)} folds.")

## 8. Backtest: Does Prediction Accuracy Translate to Returns?RMSE and MAPE tell you about price-prediction error. They do **not** directly tell you whether a trading strategy based on the model would have made money. This section runs a minimal long/flat backtest: go long when the model predicts tomorrow's price is higher than today's, otherwise hold cash — including realistic transaction costs — and compares the result to simply buying and holding.

In [ ]:
test_dates = test_df.index[LOOKBACK_WINDOW:]lstm_backtest = backtest_directional_strategy(    lstm_actual, lstm_pred, test_dates, transaction_cost_bps=TRANSACTION_COST_BPS)attn_backtest = backtest_directional_strategy(    attn_actual, attn_pred, test_dates, transaction_cost_bps=TRANSACTION_COST_BPS)buy_hold = backtest_buy_and_hold(lstm_actual, test_dates)print("=== LSTM-driven strategy ===")print(f"  Total return:      {lstm_backtest.total_return_pct:.2f}%")print(f"  Annualized return: {lstm_backtest.annualized_return_pct:.2f}%")print(f"  Max drawdown:      {lstm_backtest.max_drawdown_pct:.2f}%")print(f"  Sharpe ratio:      {lstm_backtest.sharpe_ratio:.2f}")print(f"  Trades:            {lstm_backtest.n_trades}  (win rate {lstm_backtest.win_rate_pct:.1f}%)")print()print("=== LSTM+Attention-driven strategy ===")print(f"  Total return:      {attn_backtest.total_return_pct:.2f}%")print(f"  Annualized return: {attn_backtest.annualized_return_pct:.2f}%")print(f"  Max drawdown:      {attn_backtest.max_drawdown_pct:.2f}%")print(f"  Sharpe ratio:      {attn_backtest.sharpe_ratio:.2f}")print(f"  Trades:            {attn_backtest.n_trades}  (win rate {attn_backtest.win_rate_pct:.1f}%)")print()print("=== Buy & Hold benchmark ===")print(f"  Total return:      {buy_hold.total_return_pct:.2f}%")print(f"  Max drawdown:      {buy_hold.max_drawdown_pct:.2f}%")print(f"  Sharpe ratio:      {buy_hold.sharpe_ratio:.2f}")

In [ ]:
plt.figure(figsize=(14, 5))plt.plot(lstm_backtest.equity_curve, label='LSTM strategy')plt.plot(attn_backtest.equity_curve, label='LSTM+Attention strategy')plt.plot(buy_hold.equity_curve, label='Buy & hold', linestyle='--', color='black')plt.title(f'{PRIMARY_TICKER} — Equity Curves (starting capital = 1.0)')plt.xlabel('Date')plt.ylabel('Equity (multiple of starting capital)')plt.legend()plt.savefig('outputs_equity_curves.png', dpi=120)plt.show()

## 9. Multi-Ticker ComparisonA model that "works" on a single stock is much less convincing than one that's at least directionally consistent across multiple tickers. Here we repeat the train/test evaluation (single split, not full walk-forward, to keep runtime reasonable) across a small basket of stocks.

In [ ]:
multi_data = fetch_multi_ticker_data(TICKERS, START_DATE, END_DATE)

In [ ]:
multi_results = []for ticker, raw in multi_data.items():    try:        feat = engineer_features(raw)        tr, va, te = chronological_split(feat, train_frac=TRAIN_FRAC, val_frac=VAL_FRAC)        sc = fit_scaler(tr, DEFAULT_FEATURE_COLS)        Xtr, ytr = build_sequences(tr, DEFAULT_FEATURE_COLS, 'Adj Close', sc, LOOKBACK_WINDOW)        Xte, yte = build_sequences(te, DEFAULT_FEATURE_COLS, 'Adj Close', sc, LOOKBACK_WINDOW)        m = build_lstm_model((LOOKBACK_WINDOW, len(DEFAULT_FEATURE_COLS)))        m.fit(Xtr, ytr, epochs=30, batch_size=BATCH_SIZE, verbose=0)        pred_scaled = m.predict(Xte, verbose=0).flatten()        pred_price = inverse_transform_column(pred_scaled, sc, target_idx, len(DEFAULT_FEATURE_COLS))        actual_price = inverse_transform_column(yte, sc, target_idx, len(DEFAULT_FEATURE_COLS))        baseline_scaled = np.roll(yte, 1); baseline_scaled[0] = yte[0]        baseline_price = inverse_transform_column(baseline_scaled, sc, target_idx, len(DEFAULT_FEATURE_COLS))        from sklearn.metrics import mean_squared_error        rmse = np.sqrt(mean_squared_error(actual_price, pred_price))        b_rmse = np.sqrt(mean_squared_error(actual_price, baseline_price))        multi_results.append({            'ticker': ticker, 'model_rmse': rmse, 'baseline_rmse': b_rmse,            'beats_baseline': rmse < b_rmse,        })        print(f"{ticker}: model RMSE=${rmse:.2f}  baseline RMSE=${b_rmse:.2f}  "              f"{'BEATS' if rmse < b_rmse else 'loses to'} baseline")    except ValueError as e:        print(f"{ticker}: skipped — {e}")multi_summary = pd.DataFrame(multi_results)multi_summary

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))x = np.arange(len(multi_summary))width = 0.35ax.bar(x - width/2, multi_summary['model_rmse'], width, label='Model RMSE')ax.bar(x + width/2, multi_summary['baseline_rmse'], width, label='Baseline RMSE')ax.set_xticks(x)ax.set_xticklabels(multi_summary['ticker'])ax.set_ylabel('RMSE ($)')ax.set_title('Model vs. Baseline RMSE Across Tickers')ax.legend()plt.savefig('outputs_multi_ticker_rmse.png', dpi=120)plt.show()n_tickers_beating = multi_summary['beats_baseline'].sum()print(f"Model beat the naive baseline on {n_tickers_beating} out of {len(multi_summary)} tickers.")

## 10. Summary & Honest ConclusionsThis notebook demonstrates a complete, methodologically sound pipeline for evaluating deep learning models on stock price prediction — and the rigor added (walk-forward validation, baseline comparison, backtesting, multi-ticker testing) is precisely what most tutorial-style stock prediction notebooks skip.**What the results above actually mean:**- If the model beats the naive baseline consistently across folds and tickers, that's a genuinely interesting (though still not tradeable-on-its-own) result.- If the model loses to the baseline in most folds/tickers — which is the most common outcome for next-day price-level prediction on liquid, efficiently-priced stocks — that is itself the correct and expected finding, not a failure of the code. Predicting next-day price levels from price history alone is extremely difficult precisely because liquid markets price in available information quickly.- The backtest result (Section 8) is the most important sanity check: even a model with seemingly reasonable RMSE can produce a losing trading strategy once transaction costs are included, especially if its directional accuracy is close to a coin flip.**Suggested next steps if continuing this project:**- Predict returns/direction rather than absolute price level (often more tractable and more directly tied to trading decisions)- Add cross-sectional features (sector momentum, market-wide volatility/VIX-like indicators)- Compare against classical statistical baselines (ARIMA, GARCH) in addition to the naive baseline- If pursuing this for trading rather than research, account for slippage, position sizing, and risk limits — none of which are modelled here